# Phase 3 — Feature Engineering & Preprocessing

Engineer new features, split data into train/test sets, and build the preprocessing pipeline (scaling + encoding) for model training.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import os

# Load cleaned dataset
df = pd.read_csv('../data/telco_churn_cleaned.csv')

# Import and apply feature engineering
from feature_helpers import engineer_features
df = engineer_features(df)

print(f"Shape after feature engineering: {df.shape}")
print(f"\nNew columns: ServiceCount, HasInternet, HasPhone, AvgMonthlyCharge, TenureGroup")
print(f"\nSample of new features:")
df[['tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'HasInternet', 'HasPhone', 'AvgMonthlyCharge', 'TenureGroup']].head(10)

Shape after feature engineering: (7032, 25)

New columns: ServiceCount, HasInternet, HasPhone, AvgMonthlyCharge, TenureGroup

Sample of new features:


,tenure,MonthlyCharges,TotalCharges,ServiceCount,HasInternet,HasPhone,AvgMonthlyCharge,TenureGroup
0,1,29.85,29.85,1,1,0,29.850000,0
1,34,56.95,1889.50,2,1,1,55.573529,2
2,2,53.85,108.15,2,1,1,54.075000,0
3,45,42.30,1840.75,3,1,0,40.905556,2
4,2,70.70,151.65,0,1,1,75.825000,0
5,8,99.65,820.50,3,1,1,102.562500,0
6,22,89.10,1949.40,2,1,1,88.609091,1
7,10,29.75,301.90,1,1,0,30.190000,0
8,28,104.80,3046.05,4,1,1,108.787500,2
9,62,56.15,3487.95,2,1,1,56.257258,3


In [2]:
# Define feature lists
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
                    'ServiceCount', 'HasInternet', 'HasPhone', 'AvgMonthlyCharge', 'TenureGroup']

categorical_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                        'PaperlessBilling', 'PaymentMethod']

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nTotal features: {len(numeric_features) + len(categorical_features)}")

# Separate features and target
X = df[numeric_features + categorical_features]
y = df['Churn']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True).round(4)}")

Numeric features (9): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'HasInternet', 'HasPhone', 'AvgMonthlyCharge', 'TenureGroup']

Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Total features: 24

X shape: (7032, 24)
y shape: (7032,)
y distribution:
Churn
0    0.7342
1    0.2658
Name: proportion, dtype: float64


In [3]:
# Train-test split (80/20) with stratification to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nTrain churn rate: {y_train.mean():.4f}")
print(f"Test churn rate:  {y_test.mean():.4f}")

X_train: (5625, 24)
X_test:  (1407, 24)

Train churn rate: 0.2658
Test churn rate:  0.2658


In [4]:
# Build preprocessing pipeline
# StandardScaler for numeric features, OneHotEncoder for categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Fit on training data only, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Extract feature names for later use (SHAP needs these)
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")
print(f"\nTotal feature columns after encoding: {len(all_feature_names)}")
print(f"\nFirst 10 feature names: {all_feature_names[:10]}")

X_train_processed shape: (5625, 35)
X_test_processed shape:  (1407, 35)

Total feature columns after encoding: 35

First 10 feature names: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'HasInternet', 'HasPhone', 'AvgMonthlyCharge', 'TenureGroup', 'gender_Male']


In [5]:
# Save the preprocessor and feature names for later use
os.makedirs('../models', exist_ok=True)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(all_feature_names, '../models/feature_names.pkl')

# Also save the split data for the next notebook
joblib.dump((X_train, X_test, y_train, y_test), '../models/train_test_split.pkl')
joblib.dump((X_train_processed, X_test_processed), '../models/processed_data.pkl')

print("Saved:")
print("  - models/preprocessor.pkl")
print("  - models/feature_names.pkl")
print("  - models/train_test_split.pkl")
print("  - models/processed_data.pkl")

Saved:
  - models/preprocessor.pkl
  - models/feature_names.pkl
  - models/train_test_split.pkl
  - models/processed_data.pkl


## Phase 3 Summary

**Feature Engineering:**
- Created 5 new features: ServiceCount, HasInternet, HasPhone, AvgMonthlyCharge, TenureGroup
- Total features: 9 numeric + 15 categorical = 24 (before encoding)

**Preprocessing Pipeline:**
- StandardScaler on 9 numeric features
- OneHotEncoder (drop='first') on 15 categorical features → 35 total columns after encoding
- Fit on training data only to prevent data leakage

**Train/Test Split:** 5,625 train / 1,407 test (80/20, stratified)

**Saved artifacts:** preprocessor.pkl, feature_names.pkl, train_test_split.pkl, processed_data.pkl